In [3]:
import pyairbnb
import pandas as pd
import concurrent.futures
import random
import time

# Configuration
PARIS_BOX = {"ne_lat": 48.902, "ne_long": 2.469, "sw_lat": 48.815, "sw_long": 2.224}

def get_listings(check_in, check_out):
    """Fetches a list of homes in the bounding box."""
    try:
        return pyairbnb.search_all(
            check_in=check_in, check_out=check_out, 
            **PARIS_BOX, zoom_value=2, currency="EUR", language="en",
            price_min=0, price_max=10000
        )
    except Exception as e:
        print(f"Search failed: {e}")
        return []

def get_details(room_id):
    """Fetches amenities for a single ID. Designed for threaded execution."""
    try:
        # Small jitter to prevent immediate IP bans during parallel requests
        time.sleep(random.uniform(0.1, 0.5))
        
        details = pyairbnb.get_details(room_id=room_id, currency="EUR", language="en")
        
        # Flatten amenities logic
        amenities = []
        raw = details.get('amenities', [])
        
        if isinstance(raw, list):
            for item in raw:
                if isinstance(item, str): 
                    amenities.append(item)
                elif isinstance(item, dict):
                    if 'title' in item: 
                        amenities.append(item['title'])
                    elif 'values' in item:
                        amenities.extend([v['title'] for v in item['values'] if v.get('available')])

        return {
            'id': str(room_id),
            'amenities_list': amenities,
            'amenity_count': len(amenities),
            'host_since': details.get('primary_host', {}).get('since')
        }
    except Exception:
        # Return empty structure on failure to keep DataFrame shape consistent
        return {'id': str(room_id), 'amenities_list': [], 'amenity_count': 0, 'host_since': None}

def run_pipeline(check_in, check_out, max_threads=5):
    print("Step 1: Scraping listings...")
    raw_results = get_listings(check_in, check_out)
    
    if not raw_results:
        print("No listings found.")
        return

    # Extract valid IDs (handling nested 'listing' keys if present)
    ids = []
    for r in raw_results:
        lid = r.get('room_id') or r.get('listing', {}).get('id')
        if lid:
            ids.append(str(lid))
    
    print(f"Step 2: Fetching details for {len(ids)} listings using {max_threads} threads...")
    
    # Parallel Execution for speed
    details_data = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_threads) as executor:
        details_data = list(executor.map(get_details, ids))

    # Step 3: Merge and Clean
    df_main = pd.json_normalize(raw_results)
    df_details = pd.DataFrame(details_data)

    # Standardize Column Names
    rename_map = {
        'room_id': 'id', 'listing.id': 'id', 
        'name': 'name', 'listing.name': 'name',
        'pricing_quote.rate.amount': 'price', 'price.amount': 'price',
        'avg_rating': 'rating', 'reviews_count': 'reviews', 
        'lat': 'latitude', 'lng': 'longitude'
    }
    df_main.rename(columns=rename_map, inplace=True)
    
    # Merge only if we have data
    if 'id' in df_main.columns and not df_details.empty:
        df_main['id'] = df_main['id'].astype(str)
        final_df = df_main.merge(df_details, on='id', how='left')

        # Filter for essential columns only
        keep_cols = ['id', 'name', 'price', 'rating', 'reviews', 'latitude', 'longitude', 
                     'amenities_list', 'amenity_count', 'host_since']
        
        # Select existing columns
        final_cols = [c for c in keep_cols if c in final_df.columns]
        final_df = final_df[final_cols]
        
        # Save
        filename = "paris_airbnb_final.csv"
        final_df.to_csv(filename, index=False)
        print(f"Done. {len(final_df)} listings saved to {filename}")
        print(final_df.head())
    else:
        print("Error: Could not merge data. Check API response structure.")

if __name__ == "__main__":
    # Run with thread pool for efficiency
    run_pipeline("2025-06-01", "2025-06-02", max_threads=8)

Step 1: Scraping listings...
Step 2: Fetching details for 280 listings using 8 threads...
Done. 324 listings saved to paris_airbnb_final.csv
                    id                                               name  \
0             29451877        Nice mezzanine at  "la dhuys" station (M11)   
1  1569015303356463273      Very cozy apartment, terrace, Montmartre view   
2             35712680                Homestay on the banks of the Ganges   
3             28295981                       Small room in a local's home   
4  1451989713243613562  Charming bright apartment in the heart of Mont...   

                                      amenities_list  amenity_count host_since  
0  [Bathroom, Bedroom and laundry, Heating and co...              8       None  
1  [Bathroom, Bedroom and laundry, Entertainment,...             10       None  
2  [Scenic views, Bathroom, Bedroom and laundry, ...             14       None  
3  [Bathroom, Bedroom and laundry, Entertainment,...             10     